In [2]:
import pandas as pd
import numpy as np

print("Libraries imported successfully.")

Libraries imported successfully.


In [6]:
rolling_plan = pd.read_csv(
    "../data/predictions/rolling_horizon_plan.csv"
)

print(
    "Rolling plan rows:",
    len(rolling_plan)
)

rolling_plan.head()

Rolling plan rows: 5


,task_id,section_id,department,estimated_duration,required_manpower,maintenance_decision_score,predicted_delay_minutes,urgency_tier,duration_slots,start_slot,end_slot,start_time,end_time
0,TMS001,NDL-MTJ-01,ENGINEERING,90,8,0.322000,0.0,MEDIUM,3,385,388,Day 9 00:30,Day 9 02:00
1,TDMS001,GWL-JHS-01,TRACTION,60,5,0.270667,0.0,MEDIUM,2,337,339,Day 8 00:30,Day 8 01:30
2,SMMS001,MTJ-AGC-01,S&T,45,3,0.274500,0.0,MEDIUM,2,340,342,Day 8 02:00,Day 8 03:00
3,TMS002,NDL-MTJ-02,ENGINEERING,60,5,0.224667,0.0,LOW,2,337,339,Day 8 00:30,Day 8 01:30
4,NEW_EVT_001,JHS-BINA-01,S&T,60,4,1.000000,60.0,CRITICAL,2,385,387,Day 9 00:30,Day 9 01:30


In [7]:
decision_data = pd.read_csv(
    "../data/predictions/multi_objective_decision_scores.csv"
)

print(
    "Decision data rows:",
    len(decision_data)
)

Decision data rows: 6


In [8]:
print(
    "Rolling plan columns:"
)

print(
    rolling_plan.columns.tolist()
)

print(
    "\nDecision data columns:"
)

print(
    decision_data.columns.tolist()
)

Rolling plan columns:
['task_id', 'section_id', 'department', 'estimated_duration', 'required_manpower', 'maintenance_decision_score', 'predicted_delay_minutes', 'urgency_tier', 'duration_slots', 'start_slot', 'end_slot', 'start_time', 'end_time']

Decision data columns:
['decision_rank', 'priority_rank', 'task_id', 'asset_id', 'section_id', 'department', 'maintenance_type', 'defect_type', 'severity', 'criticality', 'overdue_days', 'estimated_duration', 'required_manpower', 'status', 'failure_probability', 'urgency_score', 'priority_score', 'priority_category', 'predicted_delay_minutes', 'operational_impact_score', 'operational_impact_category', 'traffic_intensity', 'affected_trains_estimate', 'failure_risk_factor', 'urgency_factor', 'criticality_factor', 'overdue_factor', 'duration_factor', 'traffic_factor', 'operational_factor', 'maintenance_decision_score', 'decision_category']


In [9]:
events = pd.DataFrame([
    {
        "event_id": "EVT_001",
        "event_type": "ASSET_FAILURE",
        "section_id": "JHS-BINA-01",
        "department": "S&T",
        "severity": 10,
        "criticality": 10,
        "delay_minutes": 60
    },
    {
        "event_id": "EVT_002",
        "event_type": "TRAFFIC_INCREASE",
        "section_id": "BPL-RTM-01",
        "department": "Operations",
        "severity": 5,
        "criticality": 7,
        "delay_minutes": 30
    },
    {
        "event_id": "EVT_003",
        "event_type": "MANPOWER_REDUCTION",
        "section_id": "RTM-VAD-01",
        "department": "Engineering",
        "severity": 6,
        "criticality": 6,
        "delay_minutes": 0
    }
])

events

,event_id,event_type,section_id,department,severity,criticality,delay_minutes
0,EVT_001,ASSET_FAILURE,JHS-BINA-01,S&T,10,10,60
1,EVT_002,TRAFFIC_INCREASE,BPL-RTM-01,Operations,5,7,30
2,EVT_003,MANPOWER_REDUCTION,RTM-VAD-01,Engineering,6,6,0


In [10]:
required_event_columns = [
    "event_id",
    "event_type",
    "section_id",
    "department",
    "severity",
    "criticality"
]

missing_columns = [
    column
    for column in required_event_columns
    if column not in events.columns
]

print(
    "Missing event columns:",
    missing_columns
)

Missing event columns: []


In [11]:
events["severity"] = pd.to_numeric(
    events["severity"],
    errors="coerce"
).fillna(0)

events["criticality"] = pd.to_numeric(
    events["criticality"],
    errors="coerce"
).fillna(0)

events["event_severity_score"] = (
    0.6 * events["severity"] / 10
    +
    0.4 * events["criticality"] / 10
)

events[
    [
        "event_id",
        "event_type",
        "event_severity_score"
    ]
]


,event_id,event_type,event_severity_score
0,EVT_001,ASSET_FAILURE,1.00
1,EVT_002,TRAFFIC_INCREASE,0.58
2,EVT_003,MANPOWER_REDUCTION,0.60


In [12]:
def classify_event_impact(event_type):

    if event_type == "ASSET_FAILURE":
        return "EMERGENCY_MAINTENANCE"

    elif event_type == "TRAFFIC_INCREASE":
        return "OPERATIONAL_PRESSURE"

    elif event_type == "MANPOWER_REDUCTION":
        return "RESOURCE_CONSTRAINT"

    else:
        return "GENERAL_DISRUPTION"

In [13]:
events["impact_type"] = (
    events["event_type"]
    .apply(classify_event_impact)
)

events[
    [
        "event_id",
        "event_type",
        "impact_type"
    ]
]

,event_id,event_type,impact_type
0,EVT_001,ASSET_FAILURE,EMERGENCY_MAINTENANCE
1,EVT_002,TRAFFIC_INCREASE,OPERATIONAL_PRESSURE
2,EVT_003,MANPOWER_REDUCTION,RESOURCE_CONSTRAINT


In [14]:
def find_affected_tasks(
    plan,
    event
):

    affected = plan[
        plan["section_id"]
        ==
        event["section_id"]
    ].copy()

    return affected

In [15]:
for _, event in events.iterrows():

    affected = find_affected_tasks(
        rolling_plan,
        event
    )

    print(
        event["event_id"],
        "|",
        event["event_type"],
        "| affected tasks:",
        len(affected)
    )

EVT_001 | ASSET_FAILURE | affected tasks: 1
EVT_002 | TRAFFIC_INCREASE | affected tasks: 0
EVT_003 | MANPOWER_REDUCTION | affected tasks: 0


In [16]:
impact_rows = []

for _, event in events.iterrows():

    affected = find_affected_tasks(
        rolling_plan,
        event
    )

    impact_rows.append({
        "event_id": event["event_id"],
        "event_type": event["event_type"],
        "section_id": event["section_id"],
        "impact_type": event["impact_type"],
        "affected_task_count": len(affected),
        "severity_score": event[
            "event_severity_score"
        ]
    })

event_impact_report = pd.DataFrame(
    impact_rows
)

event_impact_report

,event_id,event_type,section_id,impact_type,affected_task_count,severity_score
0,EVT_001,ASSET_FAILURE,JHS-BINA-01,EMERGENCY_MAINTENANCE,1,1.00
1,EVT_002,TRAFFIC_INCREASE,BPL-RTM-01,OPERATIONAL_PRESSURE,0,0.58
2,EVT_003,MANPOWER_REDUCTION,RTM-VAD-01,RESOURCE_CONSTRAINT,0,0.60


In [17]:
events["rescheduling_priority"] = (
    events["event_severity_score"]
    * events["delay_minutes"].clip(
        lower=0
    ).apply(
        lambda x: 1 + x / 60
    )
)

events[
    [
        "event_id",
        "event_type",
        "rescheduling_priority"
    ]
]

,event_id,event_type,rescheduling_priority
0,EVT_001,ASSET_FAILURE,2.00
1,EVT_002,TRAFFIC_INCREASE,0.87
2,EVT_003,MANPOWER_REDUCTION,0.60


In [18]:
def calculate_task_adjustment(
    task,
    event
):

    adjustment = 0.0

    if (
        task["section_id"]
        ==
        event["section_id"]
    ):

        if event["event_type"] == "ASSET_FAILURE":

            adjustment += 1.0

        elif event["event_type"] == "TRAFFIC_INCREASE":

            adjustment -= (
                event["delay_minutes"]
                / 100
            )

        elif event["event_type"] == "MANPOWER_REDUCTION":

            adjustment -= 0.2

    return adjustment

In [19]:
adjusted_plan = rolling_plan.copy()

adjusted_plan["event_adjustment"] = 0.0

In [20]:
for _, event in events.iterrows():

    mask = (
        adjusted_plan["section_id"]
        ==
        event["section_id"]
    )

    adjusted_plan.loc[
        mask,
        "event_adjustment"
    ] += adjusted_plan.loc[
        mask
    ].apply(
        lambda task:
        calculate_task_adjustment(
            task,
            event
        ),
        axis=1
    )

In [21]:
adjusted_plan[
    "adjusted_priority"
] = (
    adjusted_plan[
        "maintenance_decision_score"
    ]
    +
    adjusted_plan[
        "event_adjustment"
    ]
)

adjusted_plan[
    "adjusted_priority"
] = (
    adjusted_plan[
        "adjusted_priority"
    ].clip(0, 1)
)

In [22]:
adjusted_plan[
    [
        "task_id",
        "section_id",
        "maintenance_decision_score",
        "event_adjustment",
        "adjusted_priority"
    ]
].sort_values(
    "adjusted_priority",
    ascending=False
).head(20)

,task_id,section_id,maintenance_decision_score,event_adjustment,adjusted_priority
4,NEW_EVT_001,JHS-BINA-01,1.000000,1.0,1.000000
0,TMS001,NDL-MTJ-01,0.322000,0.0,0.322000
2,SMMS001,MTJ-AGC-01,0.274500,0.0,0.274500
1,TDMS001,GWL-JHS-01,0.270667,0.0,0.270667
3,TMS002,NDL-MTJ-02,0.224667,0.0,0.224667


In [23]:
adjusted_plan[
    "reschedule_required"
] = (
    adjusted_plan[
        "event_adjustment"
    ].abs()
    >= 0.20
)

print(
    "Tasks requiring rescheduling:",
    adjusted_plan[
        "reschedule_required"
    ].sum()
)

Tasks requiring rescheduling: 1


In [24]:
adjusted_plan[
    "rescheduling_priority"
] = np.select(
    [
        adjusted_plan["adjusted_priority"] >= 0.80,
        adjusted_plan["adjusted_priority"] >= 0.60,
        adjusted_plan["adjusted_priority"] >= 0.40
    ],
    [
        "CRITICAL",
        "HIGH",
        "MEDIUM"
    ],
    default="LOW"
)

adjusted_plan[
    [
        "task_id",
        "adjusted_priority",
        "rescheduling_priority"
    ]
].head(20)

,task_id,adjusted_priority,rescheduling_priority
0,TMS001,0.322000,LOW
1,TDMS001,0.270667,LOW
2,SMMS001,0.274500,LOW
3,TMS002,0.224667,LOW
4,NEW_EVT_001,1.000000,CRITICAL


In [25]:
section_pressure = (
    adjusted_plan
    .groupby("section_id")
    .agg(
        affected_tasks=(
            "reschedule_required",
            "sum"
        ),
        average_priority=(
            "adjusted_priority",
            "mean"
        ),
        maximum_priority=(
            "adjusted_priority",
            "max"
        )
    )
    .reset_index()
    .sort_values(
        "maximum_priority",
        ascending=False
    )
)

section_pressure.head(15)

,section_id,affected_tasks,average_priority,maximum_priority
1,JHS-BINA-01,1,1.000000,1.000000
3,NDL-MTJ-01,0,0.322000,0.322000
2,MTJ-AGC-01,0,0.274500,0.274500
0,GWL-JHS-01,0,0.270667,0.270667
4,NDL-MTJ-02,0,0.224667,0.224667


In [26]:
rescheduling_queue = (
    adjusted_plan[
        adjusted_plan[
            "reschedule_required"
        ]
    ]
    .sort_values(
        "adjusted_priority",
        ascending=False
    )
    .copy()
)

print(
    "Rescheduling queue:",
    len(rescheduling_queue)
)

Rescheduling queue: 1


In [27]:
rescheduling_queue[
    [
        "task_id",
        "section_id",
        "department",
        "adjusted_priority",
        "rescheduling_priority"
    ]
].head(20)

,task_id,section_id,department,adjusted_priority,rescheduling_priority
4,NEW_EVT_001,JHS-BINA-01,S&T,1.0,CRITICAL


In [28]:
event_summary = (
    events
    .groupby(
        [
            "event_type",
            "impact_type"
        ]
    )
    .agg(
        event_count=("event_id", "count"),
        average_severity=(
            "event_severity_score",
            "mean"
        ),
        average_rescheduling_priority=(
            "rescheduling_priority",
            "mean"
        )
    )
    .reset_index()
)

event_summary

,event_type,impact_type,event_count,average_severity,average_rescheduling_priority
0,ASSET_FAILURE,EMERGENCY_MAINTENANCE,1,1.00,2.00
1,MANPOWER_REDUCTION,RESOURCE_CONSTRAINT,1,0.60,0.60
2,TRAFFIC_INCREASE,OPERATIONAL_PRESSURE,1,0.58,0.87


In [29]:
event_impact_report.to_csv(
    "../data/predictions/event_impact_report.csv",
    index=False
)

print(
    "Saved:",
    "../data/predictions/event_impact_report.csv"
)

Saved: ../data/predictions/event_impact_report.csv


In [30]:
adjusted_plan.to_csv(
    "../data/predictions/dynamically_adjusted_plan.csv",
    index=False
)

print(
    "Saved:",
    "../data/predictions/dynamically_adjusted_plan.csv"
)

Saved: ../data/predictions/dynamically_adjusted_plan.csv


In [31]:
print(
    "===== DYNAMIC RESCHEDULING SUMMARY ====="
)

print(
    "Total events:",
    len(events)
)

print(
    "Affected tasks:",
    adjusted_plan[
        "reschedule_required"
    ].sum()
)

print(
    "Critical rescheduling tasks:",
    (
        adjusted_plan[
            "rescheduling_priority"
        ]
        == "CRITICAL"
    ).sum()
)

print(
    "\nEvent impact:"
)

display(event_impact_report)

print(
    "\nRescheduling queue:"
)

display(
    rescheduling_queue[
        [
            "task_id",
            "section_id",
            "department",
            "adjusted_priority",
            "rescheduling_priority"
        ]
    ].head(20)
)

===== DYNAMIC RESCHEDULING SUMMARY =====
Total events: 3
Affected tasks: 1
Critical rescheduling tasks: 1

Event impact:


,event_id,event_type,section_id,impact_type,affected_task_count,severity_score
0,EVT_001,ASSET_FAILURE,JHS-BINA-01,EMERGENCY_MAINTENANCE,1,1.00
1,EVT_002,TRAFFIC_INCREASE,BPL-RTM-01,OPERATIONAL_PRESSURE,0,0.58
2,EVT_003,MANPOWER_REDUCTION,RTM-VAD-01,RESOURCE_CONSTRAINT,0,0.60



Rescheduling queue:


,task_id,section_id,department,adjusted_priority,rescheduling_priority
4,NEW_EVT_001,JHS-BINA-01,S&T,1.0,CRITICAL
